In [1]:
from src.data_generator import generate_fleet_data
from src.features import create_features, get_feature_names
from src.scoring import (
    score_features, plot_feature_scoring, plot_freq_boxplot,
    plot_feature_correlation, eliminate_correlated_features,
)
from src.models import (
    compare_models, tune_best_model,
    plot_roc_curves, plot_confusion_matrix,
    plot_loocv_performance, plot_feature_importance,
)
from xgboost import XGBClassifier
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
""" print("=" * 60)
    print("  IDG FREKANS ARIZA TAHMİNİ - ANALİZ PIPELINE")
    print("=" * 60)

    # 1. Veri üretimi
    print("\n[1/6] Sentetik filo verisi üretiliyor...")
    data = generate_fleet_data(n_aircraft=10, n_failing=3, seed=42)
    print(f"  Toplam kayıt: {len(data)}")
    print(f"  Uçak sayısı: {data['aircraft_id'].nunique()}")
    print(f"  Pozitif oran: {data['label'].mean():.2%}")"""

data = pd.read_csv("goksu_fleet_data.csv")
data.drop(columns=["flight_id"], inplace=True)

In [3]:
data.head()

,date,aircraft_id,freq_left,freq_right,label
0,2023-01-02 01:00:00,AC_1,398.4920,401.9848,0
1,2023-01-02 01:00:01,AC_1,398.2772,402.0431,0
2,2023-01-02 01:00:02,AC_1,398.3156,402.1218,0
3,2023-01-02 01:00:03,AC_1,398.1249,401.9571,0
4,2023-01-02 01:00:04,AC_1,398.0183,402.0419,0


In [4]:
# 2. Özellik mühendisliği
print("\n[2/6] Simetrik özellikler türetiliyor...")
df = create_features(data)
feature_cols = get_feature_names()
print(f"  Özellik sayısı: {len(feature_cols)}")


[2/6] Simetrik özellikler türetiliyor...
  Özellik sayısı: 9


In [5]:
df.head()

,date,aircraft_id,freq_left,freq_right,label,abs_diff,max_freq,min_freq,mean_freq,ratio,range_norm,std_pair,max_dev_from_400,sum_dev_from_400
0,2023-01-02 01:00:00,AC_1,398.4920,401.9848,0,3.4928,401.9848,398.4920,400.23840,1.008765,0.008727,1.74640,1.9848,3.4928
1,2023-01-02 01:00:01,AC_1,398.2772,402.0431,0,3.7659,402.0431,398.2772,400.16015,1.009455,0.009411,1.88295,2.0431,3.7659
2,2023-01-02 01:00:02,AC_1,398.3156,402.1218,0,3.8062,402.1218,398.3156,400.21870,1.009556,0.009510,1.90310,2.1218,3.8062
3,2023-01-02 01:00:03,AC_1,398.1249,401.9571,0,3.8322,401.9571,398.1249,400.04100,1.009626,0.009580,1.91610,1.9571,3.8322
4,2023-01-02 01:00:04,AC_1,398.0183,402.0419,0,4.0236,402.0419,398.0183,400.03010,1.010109,0.010058,2.01180,2.0419,4.0236


In [7]:
# 3. EDA ve özellik skorlama
print("\n[3/6] Özellik değerlendirme ve EDA...")
plot_freq_boxplot(data, "images/freq_boxplot.png")

score_df = score_features(df, feature_cols)
print("\n  Özellik Skorları:")
print(score_df.to_string())
plot_feature_scoring(score_df, "images/feature_scoring.png")

# Özellik korelasyon analizi ve eleme
plot_feature_correlation(df, feature_cols, "images/feature_correlation.png")
feature_cols, dropped_features = eliminate_correlated_features(df, feature_cols, score_df)
print(f"\n  Elenen özellikler (|r| > 0.90): {dropped_features}")
print(f"  Kalan özellikler ({len(feature_cols)}): {feature_cols}")

# LaTeX için kalan özellikleri satır içi snippet olarak dışa aktar
def _latex_feature_list(features):
    parts = [r"\texttt{" + f.replace("_", r"\_") + "}" for f in features]
    if len(parts) == 1:
        return parts[0]
    return ", ".join(parts[:-1]) + " ve " + parts[-1]

with open("retained_features.tex", "w", encoding="utf-8") as _fh:
    _fh.write(_latex_feature_list(feature_cols))
print("  -> Kaydedildi: retained_features.tex")


[3/6] Özellik değerlendirme ve EDA...
  -> Kaydedildi: images/freq_boxplot.png

  Özellik Skorları:
                   ROC_AUC        KS  Korelasyon  Stabilite (AUC)  Bileşik Skor
Özellik                                                                        
max_dev_from_400  1.000000  1.000000    0.901854         1.000000      0.975463
max_freq          0.767489  0.651093    0.611707         0.694238      0.681132
mean_freq         0.644857  0.582300    0.165179         0.985805      0.594535
min_freq          0.566181  0.348907    0.397178         0.694116      0.501595
  -> Kaydedildi: images/feature_scoring.png
  -> Kaydedildi: images/feature_correlation.png

  Elenen özellikler (|r| > 0.90): []
  Kalan özellikler (4): ['max_freq', 'min_freq', 'mean_freq', 'max_dev_from_400']
  -> Kaydedildi: retained_features.tex


In [8]:
# 4. Model karşılaştırma
print("\n[4/6] Model karşılaştırması (LOOCV)...")
results, comparison_df = compare_models(df, feature_cols)
print("\n  Model Karşılaştırma Tablosu:")
display_cols = ["F1", "Precision", "Recall", "Accuracy", "ROC AUC", "PR AUC"]
available_cols = [c for c in display_cols if c in comparison_df.columns]
print(comparison_df[available_cols].to_string(float_format="{:.3f}".format))
plot_roc_curves(results, "images/roc_curves.png")

# LaTeX tablo satırları olarak kaydet
with open("model_results.txt", "w", encoding="utf-8") as f:
    for model_name, row in comparison_df[available_cols].iterrows():
        values = " & ".join(f"{v:.3f}" for v in row)
        f.write(f"{model_name} & {values} \\\\\n")
print("  -> Kaydedildi: model_results.txt")



[4/6] Model karşılaştırması (LOOCV)...
  Model: XGBoost...
  Model: LightGBM...
  Model: Lojistik Regresyon...
  Model: Random Forest...
  Model: Linear SVM...
  Model: Extra Trees...

  Model Karşılaştırma Tablosu:
                      F1  Precision  Recall  Accuracy  ROC AUC  PR AUC
Model                                                                 
XGBoost            1.000      1.000   1.000     1.000    1.000   1.000
LightGBM           1.000      1.000   0.999     1.000    1.000   1.000
Lojistik Regresyon 1.000      1.000   1.000     1.000    1.000   1.000
Random Forest      1.000      1.000   1.000     1.000    1.000   1.000
Linear SVM         1.000      1.000   1.000     1.000      NaN     NaN
Extra Trees        1.000      1.000   1.000     1.000    1.000   1.000
  -> Kaydedildi: images/roc_curves.png
  -> Kaydedildi: model_results.txt


In [ ]:
# 5. Hiperparametre optimizasyonu
print("\n[5/6] Hiperparametre optimizasyonu ve eşik ayarlama...")
tuned = tune_best_model(df, feature_cols)
print(f"  En iyi parametreler: {tuned['best_params']}")

print(f"\n  --- Sadece Hiperparametre Optimizasyonu (eşik=0.5) ---")
print(f"  F1:        {tuned['base_metrics']['F1']:.3f}")
print(f"  Recall:    {tuned['base_metrics']['Recall']:.3f}")
print(f"  Precision: {tuned['base_metrics']['Precision']:.3f}")

print(f"\n  --- Hiperparametre + Eşik Optimizasyonu (eşik={tuned['best_threshold']:.2f}) ---")
print(f"  F1:        {tuned['metrics']['F1']:.3f}")
print(f"  Recall:    {tuned['metrics']['Recall']:.3f}")
print(f"  Precision: {tuned['metrics']['Precision']:.3f}")

plot_confusion_matrix(
    tuned["y_true"], tuned["y_pred"],
    "images/confusion_matrix.png"),

# Özellik önemleri
best_model = XGBClassifier(
    **tuned["best_params"],
    eval_metric="logloss", use_label_encoder=False,
    verbosity=0, random_state=42, n_jobs=-1
)
plot_feature_importance(
    best_model, feature_cols, df, feature_cols, "label",
    "images/feature_importance.png")



[5/6] Hiperparametre optimizasyonu ve eşik ayarlama...
  En iyi parametreler: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1}

  --- Sadece Hiperparametre Optimizasyonu (eşik=0.5) ---
  F1:        1.000
  Recall:    1.000
  Precision: 1.000

  --- Hiperparametre + Eşik Optimizasyonu (eşik=0.10) ---
  F1:        1.000
  Recall:    1.000
  Precision: 1.000
  -> Kaydedildi: images/confusion_matrix.png
  -> Kaydedildi: images/feature_importance.png


In [10]:
# 6. LOOCV uçak bazlı performans
print("\n[6/6] Uçak bazlı LOOCV sonuçları...")
# En iyi modelin per-aircraft sonuçlarını kullan
xgb_per_aircraft = results["XGBoost"]["per_aircraft"]
plot_loocv_performance(
    xgb_per_aircraft,
    "images/loocv_performance.png")
    


[6/6] Uçak bazlı LOOCV sonuçları...
  -> Kaydedildi: images/loocv_performance.png


In [11]:
import os

print("\n" + "=" * 60)
print("  TAMAMLANDI! Tüm grafikler images/ klasörüne kaydedildi.")
print("=" * 60)
print(f"\n  Üretilen dosyalar:")
for f in sorted(os.listdir("images")):
    print(f"    - {f}")


  TAMAMLANDI! Tüm grafikler images/ klasörüne kaydedildi.

  Üretilen dosyalar:
    - confusion_matrix.png
    - feature_importance.png
    - feature_scoring.png
    - freq_boxplot.png
    - loocv_performance.png
    - roc_curves.png
